# Okay...this is it. The Kalman Filter.

Last time, we looked at the follow filters: moving averages, low-pass filters, alpha filters, and alpha beta gamma filters. 

**All of these filters had the same general idea:**

**Use noisy measurements to make a better estimate of the true state of the rocket/vehicle.**

The Kalman Filter does the same thing, but in a more formal and mathematically rigorous way. Instead of us just randomly picking alpha, beta, gamma values and hoping they work, the Kalman Filter uses uncertainty to decide how much it should trust the model (based on physics) vs. the sensor (new measurement).

So in simple words:

- The model predicts what should happen
- The sensor measures what actually happened
- Both are imperfect
- The Kalman Filter combines them based on how uncertain each one is. 

This seems a little more logical than trusting yourself to make the judgement on how much to trust the sensor vs model right? That's why its used all over the aerospace industry! 

DISCLAIMER: This is one of those things that looks terrifying when you first see the equations, but the idea is honestly not that bad. Just hold on! 

### Let's make some noisy altitude data again...

Suppose our rocket is going upward with roughly constant velocity. Obviously this isn't how a rocket actually works but for the sake of simplicity lets assume it. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

dt = 0.1
time = np.arange(0, 20, dt)

true_position = 100 + 25 * time
measurement_noise = np.random.normal(0, 35, len(time))
measured_position = true_position + measurement_noise

plt.figure(figsize=(10, 4))
plt.plot(time, true_position, label="True Position", linewidth=3)
plt.scatter(time, measured_position, s=10, alpha=0.6, label="Measured Position")
plt.xlabel("Time (s)")
plt.ylabel("Altitude (m)")
plt.grid(True)
plt.legend()
plt.show()

Again, the true altitude is smooth but the measured altitude is noisy. This is the same case that we explored with the last module. **In real life, we do not know the true altitude, we only have the sensor measurements to go off of.**

**Now, this is the recipe: the two parts of the Kalman Filter**

A Kalman Filter repeats only two steps:

1. **Predict**
2. **Update**

The predict step says:

"Based on my physics model, where do I think I am now?"

This is purely mathematical. It may be correct it may not. 

The update step says:

"Okay now that I got a measurement and I know what my physics model, how much should I correct myself to be more aligned to where I think I actually am?"

That is really all a Kalman Filter is. See the similarity with the other filters? 

**Uncertainty**

The special thing about the Kalman Filter is that it tracks uncertainty.

If the filter is very uncertain about its prediction, it should trust the sensor more.

If the sensor is very noisy (meaning theres high uncertainty to the sensor), it should trust the prediction more.

We usually describe this with:

- $P$ = uncertainty in our estimate (where the Kalman Filter thinks we are)
- $Q$ = uncertainty in our model  (where the physics thinks we are)
- $R$ = uncertainty in our measurement  (where the sensors thinks we are)

For now, do not worry about where these numbers come from. Just understand what they mean.

**A super simple 1D Kalman Filter**

To simplify, lets estimate just one number: altitude.

We will pretend altitude does not have velocity yet. Again, this is not how a rocket works, but bear with me.  

The Kalman gain is:

$K = \frac{P}{P + R}$

where:

- $P$ is how uncertain we are in our Kalman Filter prediction
- $R$ is how uncertain we are is in our measurement prediction

Then the update is:

$estimate = prediction + K(measurement - prediction)$

Where estimate is what our filter thinks is our state, prediction is based on our physics model and measurement is what the sensors are telling us. 

Look familiar? Its basically the alpha filter, except now alpha is computed for us and changes with each iteration.

That computed alpha for the Kalman filter is called the **Kalman Gain**. If you're wondering how Kalman gain is derived I've linked the derivation here: [INSERT LINK]

The main point I want to drive is, when measurement noise $R$ is huge, $K$ gets smaller. When prediction uncertainty $P$ is huge, $K$ gets larger. This matches the intuition perfectly. A question may be is that why do we not factor in the $Q$ instead in the form $K = \frac{Q}{Q+R}$? In this case, our Kalman gain won't take into account past estimated states! Thus, we formulate it using $P$ (which takes into account both measurement and physics model of all past iterations) and it follows that the weight on our prediction is simply $(1-K)$. 


Now in python....

In [ ]:
def simple_kalman_filter(measurements, initial_estimate, initial_uncertainty, measurement_uncertainty, model_uncertainty):
    estimates = []
    gains = []

    estimate = initial_estimate
    P = initial_uncertainty
    R = measurement_uncertainty
    Q = model_uncertainty

    for z in measurements:
        # Predict
        predicted_estimate = estimate
        predicted_P = P + Q

        # Update
        K = predicted_P / (predicted_P + R)
        estimate = predicted_estimate + K * (z - predicted_estimate)
        P = (1 - K) * predicted_P

        estimates.append(estimate)
        gains.append(K)

    return np.array(estimates), np.array(gains)

kalman_position, kalman_gain = simple_kalman_filter(
    measured_position,
    initial_estimate=measured_position[0],
    initial_uncertainty=100,
    measurement_uncertainty=25**2,
    model_uncertainty=5
)

plt.figure(figsize=(10, 4))
plt.scatter(time, measured_position, s=10, alpha=0.5, label="Measured Position")
plt.plot(time, kalman_position, label="1D Kalman Estimate", linewidth=3)
plt.plot(time, true_position, label="True Position", linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Altitude (m)")
plt.grid(True)
plt.legend()
plt.show()

This already looks better than raw measurements.

But notice something important: this filter does not really understand motion yet. It is smoothing altitude, but it does not know the rocket has velocity.

For that, we need the matrix form.

**What do Q and R actually do?**

This is probably the most important tuning idea.

$R$ is measurement noise.

If $R$ is big, the filter trusts the sensor less.

$Q$ is model noise.

If $Q$ is big, the filter trusts the model less.

So:

- Bigger $R$ -> smoother estimate, less sensor trust
- Smaller $R$ -> noisier estimate, more sensor trust
- Bigger $Q$ -> reacts faster, less model trust
- Smaller $Q$ -> reacts slower, more model trust

This tuning can take hours on end....Kalman Filter tuning is no easy task.

### 🎯 Your Turn

Change `measurement_uncertainty` and `model_uncertainty` in the cell below.

Try:

- Larger measurement uncertainty
- Smaller measurement uncertainty
- Larger model uncertainty

What happens to the estimate?

In [ ]:
MEASUREMENT_UNCERTAINTY = 35**2
MODEL_UNCERTAINTY = 5

kalman_position, kalman_gain = simple_kalman_filter(
    measured_position,
    initial_estimate=measured_position[0],
    initial_uncertainty=100,
    measurement_uncertainty=MEASUREMENT_UNCERTAINTY,
    model_uncertainty=MODEL_UNCERTAINTY
)

plt.figure(figsize=(10, 4))
plt.scatter(time, measured_position, s=10, alpha=0.5, label="Measured Position")
plt.plot(time, kalman_position, label="Kalman Estimate", linewidth=3)
plt.xlabel("Time (s)")
plt.ylabel("Altitude (m)")
plt.grid(True)
plt.legend()
plt.show()

### The Matrix Kalman Filter 

Okay now lets do it the way you will actually see it in GNC.

We define our state as:

$x = \begin{bmatrix} position \\ velocity \end{bmatrix} = \begin{bmatrix} s_x \\ s_y \\ s_z \\ \dot{s_x} \\ \dot{s_y}\\ \dot{s_z} \end{bmatrix}$

So instead of estimating only altitude, we estimate altitude and velocity together. This is called our **state vector**. 

You may have never encountered Newtons notation before. Thats okay! A dot simply means "the derivative of". The derivative of position ($s_x,s_y,s_z$) is velocity ($\dot{s_x},\dot{s_y},\dot{s_z}$).

This is useful because sensors may only measure position, but the filter can still estimate velocity from how position changes over time. The state vector is basically what you're interested in knowing. 

**Prediction Step**

For constant velocity motion:

$position_{new} = position_{old} + velocity \Delta t$ (1)
 
$velocity_{new} = velocity_{old}$ (2)

In matrix form:

$x_{pred} = F x$

where:

$F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}$

$F$ is a **state transition matrix**. It takes the equations (1) and (2) and puts them into matrix form, such that we can multiply by our measurement (position) and get out something in the form of our state vector. 

**Measurement Step**

Our sensor only measures position.

So the measurement matrix is:

$H = \begin{bmatrix} 1 & 0 \end{bmatrix}$

This basically says:

"Take the position part of the state and ignore velocity." It gets us into the form of the measurement given we know a state vector. 

**The Kalman Filter equations**

Here are the equations. They look like alot, but each one has a job.

Prediction:

$x_{pred} = Fx$ (produce our physics based state prediction)

$P_{pred} = FP F^T + Q$ (propagate how confident we currently are in our estimate after factoring in the physics model)

Update:

$y = z - Hx_{pred}$ (produce our residual, like we saw in the alpha filter)

$S = HP_{pred}H^T + R$ (produce the uncertainty of our measurement)

$K = P_{pred}H^TS^{-1}$ (produce a Kalman gain)

$x = x_{pred} + Ky$ (like the alpha filter, produce a new state with both the physics and measurement accounted for.)

$P = (I - KH)P_{pred}$ (propagate how confident we currently are in our estimate after factoring in the measurement)

where:

- $x$ is the state estimate
- $P$ is estimate uncertainty
- $Q$ is model uncertainty
- $R$ is measurement uncertainty
- $K$ is the Kalman gain
- $y$ is the residual, aka measurement error

Again, do not panic. We will code it line by line.

**Okay but why the matrix equations?**

The matrix version is doing the exact same thing we saw with the 1D, but now our state has multiple values.

$P$ tells us:

- How uncertain position is
- How uncertain velocity is
- How position and velocity uncertainty are related

That last part is why covariance matrices matter (tells us error relationships between variables). The filter is not just tracking error size. It is tracking how errors move together. In essence P is a covariance matrix for all of the stats nerds out there. 

$P = \begin{bmatrix} \sigma^2_{position} & \sigma_{position, velocity} \\ \sigma_{velocity, position} & \sigma^2_{velocity} \end{bmatrix}$


### Why is $P_{pred} = FPF^T + Q$?

Think of this as **propagating our uncertainty through the physics model**.

We start with our current state uncertainty, represented by $P$. Our physics model tells us how the state is expected to change. Since the uncertainty is attached to the state, we also need to transform that uncertainty according to how the physics model transforms the state. This gives:

$FPF^T$

In other words, **we take our current uncertainty and transform it into the uncertainty of the predicted state according to our physics model**.

However, our physics model is not perfect. There are effects that we do not model exactly, such as disturbances, approximations, or unknown forces. We represent this additional uncertainty with the process noise covariance (Q).

Therefore, the predicted uncertainty is:


$P_{pred} = FPF^T + Q$

Q is what actually changes our state uncertainty in this step. 


So conceptually:

$
\boxed{\text{Current uncertainty}
\rightarrow
\text{transform through physics model}
\rightarrow
\text{add uncertainty from model imperfections}}
$

The key idea is that **(FPF^T) tells us how our existing uncertainty evolves according to the physics model, while (Q) accounts for uncertainty introduced by the physics model itself.**


## Why is $K = P_{pred}H^TS^{-1}$?

Similar to $P_{pred} = FPF^T + Q$, the measurement prediction is:

$z_{pred} = Hx_{pred}$

The residual is:

$y = z - Hx_{pred}$

The uncertainty of that residual is:

$S = HP_{pred}H^T + R$

This says:

- $HP_{pred}H^T$ is the uncertainty in the predicted measurement
- $R$ is the uncertainty in the actual measurement

This is remarkably similar to how we understood the physics model uncertainty contribution. 

The Kalman Gain in matrix form becomes:

$K = P_{pred}H^TS^{-1}$

This is the matrix version of:

$K = \frac{P}{P + R}$

You can even see the similarity:

- $P_{pred}H^T$ is like the numerator
- $S = HP_{pred}H^T + R$ is like the denominator

So the Kalman gain is still answering the same question we looked at in the 1D case just now in linear algebra:

**How much should I trust the measurement correction?**

## Why is $P = (I - KH)P_{pred}$? 

After the update, the estimate is:

$x = x_{pred} + Ky$

where:

$y = z - Hx_{pred}$

The term $KH$ tells us how much of the predicted state got corrected by the measurement.

So $(I - KH)$ tells us how much uncertainty is left after the correction. We just factor this into our current state. 

That gives:

$P = (I - KH)P_{pred}$

I want to add in, there is a longer form called the Joseph form:

$P = (I-KH)P_{pred}(I-KH)^T + KRK^T$

This is more numerically stable in real flight software, but for learning the shorter form is easier to understand! 

Broadly:

- Prediction usually increases uncertainty.
- Measurement update usually decreases uncertainty.
- The Kalman Gain decides how aggressive that decrease should be. 
- Overtime, your confidence in your current state should stabilize. 

Now in python....

In [ ]:
def kalman_filter_position_velocity(measurements, dt, measurement_std):
    states = []
    residuals = []
    uncertainty_P = []

    x = np.array([[measurements[0]],
                  [0]])

    P = np.array([[100, 0],
                  [0, 100]])

    F = np.array([[1, dt],
                  [0, 1]])

    H = np.array([[1, 0]])

    Q = np.array([[1, 0],
                  [0, 3]])

    R = np.array([[measurement_std**2]])

    I = np.eye(2)

    for z in measurements:
        z = np.array([[z]])

        # Predict
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        # Update
        y = z - H @ x_pred
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (I - K @ H) @ P_pred

        states.append(x.flatten())
        residuals.append(y.item())
        uncertainty_P.append(P.flatten())

    return np.array(states), np.array(residuals), np.array(uncertainty_P)

states, residuals, uncertainty_P = kalman_filter_position_velocity(
    measured_position,
    dt=dt,
    measurement_std=35
)

estimated_position = states[:, 0]
estimated_velocity = states[:, 1]
position_uncertainty = uncertainty_P[:, 0]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.scatter(time, measured_position, s=10, alpha=0.5, label="Measured Position")
ax1.plot(time, true_position, label="True Position", linewidth=2)
ax1.plot(time, estimated_position, label="Kalman Position", linewidth=3)
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Altitude (m)")
ax1.grid(True)
ax1.legend()

ax2.plot(time, estimated_velocity, label="Kalman Velocity", linewidth=3)
ax2.axhline(25, color="black", linestyle="--", label="True Velocity")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Velocity (m/s)")
ax2.grid(True)
ax2.legend()

ax3.plot(time, position_uncertainty, label="Position Uncertainty", linewidth=3)
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("Position Uncertainty (m)")
ax3.grid(True)
ax3.legend()

plt.tight_layout()
plt.show()

Now this is more useful.

The sensor only measured position, but the Kalman Filter estimated position and velocity.

This is why the Kalman Filter is so powerful. It can estimate states that are not directly measured, as long as the model connects them to what we can measure.

### 🎯 Your Turn

In the filter function below, change the values inside `Q` and rerun the cells.

Try:

- Making velocity uncertainty larger
- Making position uncertainty larger
- Making both really small

What happens to position and velocity?

In [ ]:
# Write your code here
def kalman_filter_position_velocity(measurements, dt, measurement_std):
    states = []
    residuals = []
    uncertainty_P = []

    x = np.array([[measurements[0]],
                  [0]])

    P = np.array([[100, 0],
                  [0, 100]])

    F = np.array([[1, dt],
                  [0, 1]])

    H = np.array([[1, 0]])

    Q = np.array([[1, 0],
                  [0, 3]])

    R = np.array([[measurement_std**2]])

    I = np.eye(2)

    for z in measurements:
        z = np.array([[z]])

        # Predict
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        # Update
        y = z - H @ x_pred
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (I - K @ H) @ P_pred

        states.append(x.flatten())
        residuals.append(y.item())
        uncertainty_P.append(P.flatten())

    return np.array(states), np.array(residuals), np.array(uncertainty_P)

states, residuals, uncertainty_P = kalman_filter_position_velocity(
    measured_position,
    dt=dt,
    measurement_std=35
)

estimated_position = states[:, 0]
estimated_velocity = states[:, 1]
position_uncertainty = uncertainty_P[:, 0]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.scatter(time, measured_position, s=10, alpha=0.5, label="Measured Position")
ax1.plot(time, true_position, label="True Position", linewidth=2)
ax1.plot(time, estimated_position, label="Kalman Position", linewidth=3)
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Altitude (m)")
ax1.grid(True)
ax1.legend()

ax2.plot(time, estimated_velocity, label="Kalman Velocity", linewidth=3)
ax2.axhline(25, color="black", linestyle="--", label="True Velocity")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Velocity (m/s)")
ax2.grid(True)
ax2.legend()

ax3.plot(time, position_uncertainty, label="Position Uncertainty", linewidth=3)
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("Position Uncertainty (m)")
ax3.grid(True)
ax3.legend()

plt.tight_layout()
plt.show()

### 🚀 Challenge: Tune the Kalman Filter

A rocket altitude sensor gives the noisy measurements below.

Your Tasks:

- Use the position/velocity Kalman Filter (write this yourself)
- Plot measured altitude, estimated altitude, and true altitude
- Plot estimated velocity and true velocity
- Can you also estimate acceleration? 
- Change `measurement_std`
- Change the values inside `Q`
- Try to make the estimate smooth but still responsive

Questions to think about:

- What happens when measurement noise is too small?
- What happens when measurement noise is too large?
- What happens when model noise is too small?

There is not one perfect answer. This is tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(12)

dt = 0.1
time = np.arange(0, 25, dt)
true_position = 20 + 8 * time + 0.5 * 2.5 * time**2
true_velocity = 8 + 2.5 * time
measured_position = true_position + np.random.normal(0, 50, len(time))

# Write your code here

**Summary**

A Kalman Filter is a filter that combines a model and measurements (sensor fusion) using uncertainty as the weights.

The summary is:

- Predict with the model
- Track uncertainty
- Use the Kalman Gain to decide how much to correct
- Update with the measurement

If you understand the predict/update loop, you understand the heart of the Kalman Filter. The math is just a way to get there. 

Next is where things get more real: what happens when the rocket model is nonlinear? A real rocket doesn't behave simply. Its dynamics can depend on things like velocity, altitude, gravity, and changing forces, creating nonlinear relationships between the states. That is where the Extended Kalman Filter comes in.